# Building RAG and Multi-Tiered AI Workflows with OpenAI (Mistral Lesson Adaptation)

In [1]:
import os
import requests
import numpy as np 
import faiss
import tiktoken
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

## Part 1: Retrieval-Augmented Generation (RAG) using OpenAI Embeddings & FAISS

In [2]:
response = requests.get('https://raw.githubusercontent.com/run-llama/llama_index/main/docs/docs/examples/data/paul_graham/paul_graham_essay.txt')

text = response.text

chunk_size=2048
chunks=[text[i:i + chunk_size] for i in range(0,len(text),chunk_size)]
print(f"Total chunks created: {len(chunks)}")

embed_model = "text-embedding-3-small"
response_embed= client.embeddings.create(
    input = chunks,
    model = embed_model
)

text_embeddings=np.array([item.embedding for item in response_embed.data])

d = text_embeddings.shape[1]
index = faiss.IndexFlatL2(d)
index.add(text_embeddings)
print("vector store indexed successfully!")



Total chunks created: 1
vector store indexed successfully!


In [3]:
question = "What did the author work on before college?"

question_embed_response = client.embeddings.create(
    input=[question],
    model= embed_model
)

question_embedding= np.array(question_embed_response.data[0].embedding)

D,I = index.search(question_embedding.reshape(1,-1),k=2)

retrieved_chunks = [chunks[i] for i in I.tolist()[0]]

prompt = f"""
Context information is below.
---------------------
{retrieved_chunks}
---------------------
Given the context information and not prior knowledge, answer the query.
Query: {question}
Answer:
"""

chat_response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3,
    max_tokens=1000
    
)

print(chat_response.choices[0].message.content)


Based on the provided context information, there is no relevant information regarding what the author worked on before college. The context indicates a "404: Not Found" error, suggesting that the requested information is not available. Therefore, I cannot provide an answer to the query.


## Part 2: Comparing Model Tiers (Fast/Cost-Effective vs. Advanced Reasoning)

In [4]:
response_small = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": "Can you write a Python function for the FizzBuzz test?"}
    ],
    max_tokens=500
)

print("---[gpt-4o-mini Response:]----")
print(response_small.choices[0].message.content)

---[gpt-4o-mini Response:]----
Certainly! The FizzBuzz test is a common coding challenge where you print numbers from 1 to a specified limit, but for multiples of 3, you print "Fizz" instead of the number, for multiples of 5 you print "Buzz", and for numbers that are multiples of both 3 and 5, you print "FizzBuzz".

Here is a Python function that implements the FizzBuzz test:

```python
def fizz_buzz(n):
    for i in range(1, n + 1):
        if i % 3 == 0 and i % 5 == 0:
            print("FizzBuzz")
        elif i % 3 == 0:
            print("Fizz")
        elif i % 5 == 0:
            print("Buzz")
        else:
            print(i)

# Example usage:
fizz_buzz(100)
```

You can call the `fizz_buzz` function with any integer `n` to see the output for numbers from 1 to `n`. In the example provided, it will print the FizzBuzz sequence from 1 to 100.


In [5]:
response_large= client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": "Can you write a Python function for the FizzBuzz test?"}
    ],
    max_tokens=500

)

print("---[gpt-4o Response]---")

print(response_large.choices[0].message.content)

---[gpt-4o Response]---
Certainly! The FizzBuzz test is a classic programming problem often used in coding interviews. Here’s a Python function that prints the numbers from 1 to `n`. For multiples of three, it prints "Fizz" instead of the number and for multiples of five, it prints "Buzz". For numbers that are multiples of both three and five, it prints "FizzBuzz":

```python
def fizz_buzz(n):
    for i in range(1, n + 1):
        if i % 3 == 0 and i % 5 == 0:
            print("FizzBuzz")
        elif i % 3 == 0:
            print("Fizz")
        elif i % 5 == 0:
            print("Buzz")
        else:
            print(i)

# Example usage:
fizz_buzz(15)
```

Running `fizz_buzz(15)` would print the following output:

```
1
2
Fizz
4
Buzz
Fizz
7
8
Fizz
Buzz
11
Fizz
13
14
FizzBuzz
```


## Part 3: Tokenization & Native Function Calling Comparison

In [6]:
encoding = tiktoken.encoding_for_model("gpt-4o")

prompt_text = "What is the weather like today in Paris ?"
tokens = encoding.encode(prompt_text)

print(f"Prompt:'{prompt_text}'")
print(f"Total Token Count:{len(tokens)}")

Prompt:'What is the weather like today in Paris ?'
Total Token Count:9
